# Notebook 03 — Leakage-Safe Dataset Splitting

## Objective

This notebook creates reproducible, leakage-safe train/validation/test assignments
for the processed MindBigData2023 dataset, and establishes a permutation-test
baseline that every future real model must be compared against.

This notebook works entirely against the local files produced by Notebook 02 —
no streaming or network access required.

---

## Tasks

1. Verify Hugging Face's official train/test split is itself leakage-safe (no
   session overlap) before trusting it.
2. Split the train pool into train/val using a block-aware (not trial-level)
   strategy, to avoid leaking paired blank/digit trials or adjacent-in-time
   trials across the boundary.
3. Validate the resulting split: zero session/block overlap, reasonable class
   balance.
4. Write split assignments back into the HDF5 files.
5. Build a reusable permutation-test (shuffled-label) harness as the floor
   every future model result must clear.

---

## Output

Both HDF5 files gain a new `split` dataset:

```
data/processed/mindbigdata2023_train.h5   -> split: "train" or "val" per trial
data/processed/mindbigdata2023_test.h5    -> split: "test" for every trial
```

In [6]:
import h5py
import numpy as np
from pathlib import Path
from collections import defaultdict

TRAIN_POOL_PATH = Path("../data/processed/mindbigdata2023_train_pool.h5")
TEST_PATH = Path("../data/processed/mindbigdata2023_test.h5")

VAL_FRACTION = 0.20   # fraction of the train pool held out as validation
SEED = 42
rng = np.random.default_rng(SEED)

## 2. Test Set Sanity Check

Before trusting Hugging Face's official train/test boundary, verify there's no
`sessionnum` overlap between our train pool and test file. If sessions are
cleanly separated, their split is already leakage-safe at the session level and
we can adopt it as our final test set. If overlap is found, we treat their
split as untrustworthy and fall back to deriving our own 3-way split (not
implemented below — stop and revisit Section 3's logic if this fires).

In [7]:
with h5py.File(TRAIN_POOL_PATH, "r") as f:
    train_sessions = set(f["sessionnum"][:])

with h5py.File(TEST_PATH, "r") as f:
    test_sessions = set(f["sessionnum"][:])

overlap = train_sessions & test_sessions

print(f"Train pool sessions: {len(train_sessions)}")
print(f"Test sessions: {len(test_sessions)}")
print(f"Overlapping sessions: {len(overlap)}")

if overlap:
    print(f"\nWARNING: {len(overlap)} sessions appear in both train and test.")
    print("Do not proceed with the rest of this notebook as written —")
    print("the official test split cannot be trusted as leakage-safe.")
    print(f"Overlapping session IDs (up to 20 shown): {sorted(overlap)[:20]}")
else:
    print("\nNo session overlap. HF's train/test boundary is leakage-safe.")
    print("Proceeding to build our own train/val split within the train pool.")

Train pool sessions: 67
Test sessions: 6
Overlapping sessions: 0

No session overlap. HF's train/test boundary is leakage-safe.
Proceeding to build our own train/val split within the train pool.


## 3. Train/Val Split — Block-Aware Assignment

We split at the **block level**, not the trial level. An entire block
(`sessionnum`, `blocknum`) — including every blank/digit pair inside it — is
assigned entirely to `train` or `val`, never split across both. This prevents
two failure modes:

- A digit trial in `val` whose paired blank trial is in `train` (or vice versa)
- Temporally adjacent trials (same block) ending up on both sides of the split,
  which could let a model exploit block-local noise/drift rather than genuine
  signal

Block assignment is randomized (fixed seed for reproducibility), then we
check the resulting class balance — we don't force-stratify at the block
level, since blocks are already balanced blank/digit by construction from
Notebook 02's pairing logic.

In [8]:
with h5py.File(TRAIN_POOL_PATH, "r") as f:
    sessionnum = f["sessionnum"][:]
    blocknum = f["blocknum"][:]
    label_digit = f["label_digit"][:]
    n_trials = len(sessionnum)

# Build the set of unique blocks present in the train pool
block_ids = np.array(list(zip(sessionnum, blocknum)))
unique_blocks = sorted(set(map(tuple, block_ids)))
print(f"Total trials: {n_trials}")
print(f"Unique blocks: {len(unique_blocks)}")

# Randomly assign each block to train or val
block_to_split = {}
for block in unique_blocks:
    block_to_split[block] = "val" if rng.random() < VAL_FRACTION else "train"

# Map each trial to its block's split assignment
trial_splits = np.array([
    block_to_split[(s, b)] for s, b in zip(sessionnum, blocknum)
])

n_train = (trial_splits == "train").sum()
n_val = (trial_splits == "val").sum()
print(f"\nTrain trials: {n_train} ({n_train/n_trials:.1%})")
print(f"Val trials:   {n_val} ({n_val/n_trials:.1%})")

Total trials: 28000
Unique blocks: 3004

Train trials: 22268 (79.5%)
Val trials:   5732 (20.5%)


## 4. Split Validation

Two checks before trusting this split:

1. **Zero block overlap** — no `(sessionnum, blocknum)` pair should appear in
   both `train` and `val` (guaranteed by construction above, but verified
   explicitly rather than assumed).
2. **Class balance per split** — both binary and digit-level distributions
   should be reasonably proportional across train/val; large skew would
   suggest the random block assignment got unlucky and should be re-seeded.

In [9]:
# Check 1: zero block overlap (sanity check on the assignment logic itself)
train_blocks = {b for b, s in block_to_split.items() if s == "train"}
val_blocks = {b for b, s in block_to_split.items() if s == "val"}
block_overlap = train_blocks & val_blocks
print(f"Block overlap between train/val: {len(block_overlap)} (should be 0)")

# Check 2: class balance per split
print("\nDigit class distribution per split:")
for split_name in ["train", "val"]:
    mask = trial_splits == split_name
    print(f"\n  {split_name} (n={mask.sum()}):")
    for d in range(-1, 10):
        count = ((label_digit == d) & mask).sum()
        pct = count / mask.sum() * 100
        print(f"    {d}: {count} ({pct:.1f}%)")

Block overlap between train/val: 0 (should be 0)

Digit class distribution per split:

  train (n=22268):
    -1: 11134 (50.0%)
    0: 1107 (5.0%)
    1: 1100 (4.9%)
    2: 1144 (5.1%)
    3: 1122 (5.0%)
    4: 1097 (4.9%)
    5: 1093 (4.9%)
    6: 1117 (5.0%)
    7: 1136 (5.1%)
    8: 1104 (5.0%)
    9: 1114 (5.0%)

  val (n=5732):
    -1: 2866 (50.0%)
    0: 293 (5.1%)
    1: 300 (5.2%)
    2: 256 (4.5%)
    3: 278 (4.8%)
    4: 303 (5.3%)
    5: 307 (5.4%)
    6: 283 (4.9%)
    7: 264 (4.6%)
    8: 296 (5.2%)
    9: 286 (5.0%)


## 5. Write Split Assignments Back to Files

We add a `split` dataset to both HDF5 files:

- `mindbigdata2023_train.h5` gets `"train"` / `"val"` per trial
- `mindbigdata2023_test.h5` gets `"test"` for every trial (uniform, but stored
  the same way for a consistent interface across both files)

In [10]:
def split_into_files(source_path, split_assignments, output_paths):
    """
    source_path: original combined train-pool file to read from
    split_assignments: array of split labels, one per trial, same order as source
    output_paths: dict mapping split name -> output file path
    """
    with h5py.File(source_path, "r") as src:
        keys = list(src.keys())   # eeg, label_binary, label_digit, metadata — no "split" here

        for split_name, out_path in output_paths.items():
            mask = split_assignments == split_name
            n = mask.sum()
            print(f"Writing {n} trials to {out_path}")

            with h5py.File(out_path, "w") as dst:
                for key in keys:
                    dst.create_dataset(
                        key,
                        data=src[key][:][mask],
                        compression="gzip" if key == "eeg" else None
                    )

output_paths = {
    "train": Path("../data/processed/mindbigdata2023_train.h5"),
    "val": Path("../data/processed/mindbigdata2023_val.h5"),
}
split_into_files(TRAIN_POOL_PATH, trial_splits, output_paths)

print("\nDone. mindbigdata2023_test.h5 requires no changes — already final from Notebook 02.")

Writing 22268 trials to ../data/processed/mindbigdata2023_train.h5
Writing 5732 trials to ../data/processed/mindbigdata2023_val.h5

Done. mindbigdata2023_test.h5 requires no changes — already final from Notebook 02.


## 6. Permutation-Test Baseline

Before any real model touches this data, we establish the floor every future
result must clear: train a simple baseline on **real** labels, then repeat on
**shuffled** labels. If a future "real" model's accuracy isn't meaningfully
above the shuffled-label distribution, it hasn't learned anything genuine —
regardless of how high the raw accuracy number looks.

This section builds the harness as a reusable function. The demo run below
uses a trivial classifier (majority-class `DummyClassifier`) on raw flattened
EEG just to confirm the harness works correctly — this is *not* a real model
attempt, just infrastructure validation. Real baseline models belong in
Phase 5 (Notebook 04+), reusing this same function.

In [12]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

def permutation_test(model_fn, X_train, y_train, X_val, y_val, n_permutations=20, seed=SEED):
    """
    Trains model_fn() on real labels, then on n_permutations rounds of shuffled
    labels. Returns (real_accuracy, shuffled_accuracies) for comparison.

    model_fn: a zero-arg callable returning an unfitted sklearn-compatible estimator.
    """
    rng_local = np.random.default_rng(seed)

    real_model = model_fn()
    real_model.fit(X_train, y_train)
    real_acc = accuracy_score(y_val, real_model.predict(X_val))

    shuffled_accs = []
    for _ in range(n_permutations):
        y_shuffled = rng_local.permutation(y_train)
        model = model_fn()
        model.fit(X_train, y_shuffled)
        shuffled_accs.append(accuracy_score(y_val, model.predict(X_val)))

    return real_acc, np.array(shuffled_accs)


# --- Harness validation demo: majority-class dummy classifier, Stage 1 (binary) ---
TRAIN_PATH = Path("../data/processed/mindbigdata2023_train.h5")
VAL_PATH = Path("../data/processed/mindbigdata2023_val.h5")

with h5py.File(TRAIN_PATH, "r") as f:
    n_train_demo = min(500, f["eeg"].shape[0])
    X_train_demo = f["eeg"][:n_train_demo].reshape(n_train_demo, -1)
    y_train_demo = f["label_binary"][:n_train_demo]

with h5py.File(VAL_PATH, "r") as f:
    n_val_demo = min(200, f["eeg"].shape[0])
    X_val_demo = f["eeg"][:n_val_demo].reshape(n_val_demo, -1)
    y_val_demo = f["label_binary"][:n_val_demo]

real_acc, shuffled_accs = permutation_test(
    lambda: DummyClassifier(strategy="most_frequent"),
    X_train_demo, y_train_demo, X_val_demo, y_val_demo,
    n_permutations=100
)

print(f"Real-label accuracy: {real_acc:.3f}")
print(f"Shuffled-label accuracy: mean={shuffled_accs.mean():.3f}, std={shuffled_accs.std():.3f}")
print("\n(Expected: both near 0.5 here, since a majority-class dummy classifier")
print("carries no real signal either way, and Stage 1 is guaranteed 50/50 by")
print("construction. This confirms the harness runs correctly.")
print("Stage 2's version — with proper stratified sampling — comes in Phase 5.)")

Real-label accuracy: 0.500
Shuffled-label accuracy: mean=0.500, std=0.000

(Expected: both near 0.5 here, since a majority-class dummy classifier
carries no real signal either way, and Stage 1 is guaranteed 50/50 by
construction. This confirms the harness runs correctly.
Stage 2's version — with proper stratified sampling — comes in Phase 5.)


## Summary

- Verified Hugging Face's official train/test split is session-leakage-safe
  (no session overlap found) — adopted as-is, no custom 3-way split needed.
- Assigned train/val split within the train pool at the **block level**
  (`sessionnum` + `blocknum`), preventing paired blank/digit trials or
  temporally adjacent trials from crossing the split boundary.
- Confirmed zero block overlap and inspected class balance across splits.
- Materialized the split into **separate files**:

```
mindbigdata2023_train_pool.h5 # source pool, kept as audit trail
mindbigdata2023_train.h5 # final training set
mindbigdata2023_val.h5 # final validation set
mindbigdata2023_test.h5 # unchanged, already final from Notebook 02
```

- Built a reusable `permutation_test()` harness.
- **Harness validation result (Stage 1, majority-class dummy classifier):**
  real accuracy = 0.500, shuffled accuracy = 0.500 ± 0.000 — exactly as
  expected for a guaranteed-balanced binary split with no real signal in a
  dummy model. Confirms both the harness and the split's class balance are
  working correctly.
- Stage 2's version of this demo (digit-only, requires stratified sampling
  due to real class imbalance risk) is deferred to Phase 5, alongside Stage
  2's other pipeline work.

**Scope going forward: Stage 1 only.** Stage 2 is deferred until a working
Stage 1 model exists — per the feature-reuse design, Stage 2 needs Stage 1's
trained backbone as a starting point, so building it now would mean throwing
work away.

**Phase 3 complete for Stage 1.** Next: Phase 4 — EEG preprocessing pipeline
(filtering, normalization, feature extraction), Stage 1 only.